# 1. Introduction

## Evaluation of CIFAR2020 Results

**Objective:**
Compare locally reproduced α,β-CROWN verification results against the official VNN-COMP CIFAR2020 benchmark to evaluate reproducibility, agreement, runtime performance, and causes of divergence.

**Key Metrics:**
- **Verification Agreement:** Percentage of properties where local and official results match.
- **Confusion Matrix:** Cross-tabulation of verification outcomes (SAT, UNSAT, TIMEOUT).
- **Runtime Performance:** Relative execution speed and runtime differences.
- **Disagreement & Solved Breakdown:** Detailed analysis of timeout and mismatch cases.

# 2. Setup & Data Loading

Import required libraries, set display options, define the result normalization helper, and load official and local benchmark datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pd.set_option("display.max_colwidth", 120)
pd.options.display.float_format = "{:.6f}".format
plt.style.use("seaborn-v0_8-whitegrid")

def normalize_result(value):
    """Normalize verification result strings to standard SAT, UNSAT, or TIMEOUT labels."""
    text = str(value).strip().upper()
    if text == "TIMEOUT":
        return "TIMEOUT"
    if text in {"SAT", "UNSAT"}:
        return text
    return text

# Load datasets
official = pd.read_csv("../data/official/cifar2020.csv")
local = pd.read_csv("../data/experiments/csv/cifar2020.csv")

print(f"Official instances loaded: {len(official)}")
print(f"Local instances loaded   : {len(local)}")

# 3. Data Validation

Verify dataset integrity (checking row counts, missing values, duplicates, and network/specification alignment) before performing comparisons.

In [ ]:
key_cols = ["benchmark", "onnx_path", "vnnlib_path"]
all_cols = key_cols + ["total_time", "result", "solver_time"]

# Verify dataset integrity
for name, frame in [("Official", official), ("Local", local)]:
    assert frame.duplicated(subset=key_cols).sum() == 0, f"{name} dataset contains duplicate properties."
    assert frame[all_cols].isna().sum().sum() == 0, f"{name} dataset contains missing values."

assert len(official) == len(local), "Dataset row counts do not match."
assert set(official["onnx_path"]) == set(local["onnx_path"]), "Network sets do not match."
assert set(official["vnnlib_path"]) == set(local["vnnlib_path"]), "Specification sets do not match."

validation_summary = pd.DataFrame({
    "Dataset": ["Official", "Local"],
    "Instances": [len(official), len(local)],
    "Unique Networks": [official["onnx_path"].nunique(), local["onnx_path"].nunique()],
    "Unique Specs": [official["vnnlib_path"].nunique(), local["vnnlib_path"].nunique()],
})
display(validation_summary)

# 4. Merge Datasets

Merge official and local datasets on property keys (`benchmark`, `onnx_path`, `vnnlib_path`) and normalize verification labels.

In [ ]:
comparison = official.merge(
    local,
    on=key_cols,
    suffixes=("_official", "_local"),
    validate="one_to_one",
)

comparison["result_official_norm"] = comparison["result_official"].map(normalize_result)
comparison["result_local_norm"] = comparison["result_local"].map(normalize_result)

display(comparison[[
    "benchmark", "onnx_path", "vnnlib_path",
    "result_official_norm", "result_local_norm",
    "total_time_official", "total_time_local"
]].head())

# 5. Verification Agreement

Calculate overall match rate and agreement percentage between local execution and official VNN-COMP results.

In [ ]:
comparison["match"] = comparison["result_official_norm"] == comparison["result_local_norm"]

instances = len(comparison)
matches = int(comparison["match"].sum())
mismatches = instances - matches
agreement_pct = (matches / instances) * 100

print(f"Instances : {instances}")
print(f"Matches   : {matches}")
print(f"Mismatches: {mismatches}")
print(f"Agreement : {agreement_pct:.1f}%")

# 6. Confusion Matrix

Cross-tabulate verification outcomes (SAT, UNSAT, TIMEOUT) and plot a heatmap to visualize agreement.

In [ ]:
labels = [label for label in ["SAT", "UNSAT", "TIMEOUT", "UNKNOWN"] 
          if label in set(comparison["result_official_norm"]) | set(comparison["result_local_norm"])]

cm_table = pd.crosstab(
    comparison["result_local_norm"],
    comparison["result_official_norm"],
    colnames=["Official"],
    rownames=["Local"]
).reindex(index=labels, columns=labels, fill_value=0)

display(cm_table)

fig, ax = plt.subplots(figsize=(6, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_table.values, display_labels=labels)
disp.plot(cmap="Blues", ax=ax, colorbar=False, values_format="d")

ax.set_xlabel("Official Result")
ax.set_ylabel("Local Result")
ax.set_title("Verification Agreement Heatmap", fontsize=12, fontweight="bold")

for text in disp.text_.ravel():
    text.set_color("black")

fig.colorbar(disp.im_, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# 7. Mismatch Analysis

Inspect properties where local verification results differ from the official benchmark.

In [ ]:
mismatch = comparison.loc[~comparison["match"], [
    "benchmark", "onnx_path", "vnnlib_path",
    "result_official_norm", "result_local_norm",
    "total_time_official", "total_time_local"
]].copy()

mismatch["result_pair"] = mismatch["result_official_norm"] + " -> " + mismatch["result_local_norm"]
mismatch["runtime_gap"] = mismatch["total_time_local"] - mismatch["total_time_official"]

if mismatch.empty:
    print("No verification disagreements were found.")
else:
    display(mismatch)
    print("\nDetailed Mismatch Report:")
    for _, row in mismatch.iterrows():
        spec_name = row["vnnlib_path"].split("/")[-1]
        print(f"Property: {spec_name:30s} | Official: {row['result_official_norm']:7s} | Local: {row['result_local_norm']:7s} | Runtime Gap: {row['runtime_gap']:+.2f}s")

# 8. Runtime Comparison

Compare total execution time between local and official runs via statistics, scatter plots, and runtime difference distributions.

In [ ]:
comparison["runtime_difference"] = comparison["total_time_local"] - comparison["total_time_official"]
comparison["speed_ratio"] = comparison["total_time_local"] / comparison["total_time_official"]

runtime_summary = comparison[["total_time_official", "total_time_local", "runtime_difference", "speed_ratio"]].agg(["mean", "median", "min", "max"]).T
display(runtime_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Scatter plot
max_time = float(max(comparison["total_time_official"].max(), comparison["total_time_local"].max()))
axes[0].scatter(comparison["total_time_official"], comparison["total_time_local"], alpha=0.8, color="#2b5c8f")
axes[0].plot([0, max_time], [0, max_time], "--", color="red", linewidth=1, label="y = x (equal runtime)")
axes[0].set_xlabel("Official Runtime (s)")
axes[0].set_ylabel("Local Runtime (s)")
axes[0].set_title("Runtime Comparison (Scatter)", fontweight="bold")
axes[0].legend()

# Histogram
axes[1].hist(comparison["runtime_difference"], bins=20, color="#4C78A8", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("Runtime Difference: Local - Official (s)")
axes[1].set_ylabel("Count")
axes[1].set_title("Runtime Difference Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

# 9. Solved vs. Timeout Analysis

Categorize properties by resolution state (Solved vs. Timeout) to assess solver completeness.

In [ ]:
comparison["official_solved"] = comparison["result_official_norm"].isin(["SAT", "UNSAT"])
comparison["local_solved"] = comparison["result_local_norm"].isin(["SAT", "UNSAT"])

solved_breakdown = pd.DataFrame({
    "Category": [
        "Solved by both",
        "Solved only officially",
        "Solved only locally",
        "Timeout by both"
    ],
    "Count": [
        int((comparison["official_solved"] & comparison["local_solved"]).sum()),
        int((comparison["official_solved"] & ~comparison["local_solved"]).sum()),
        int((~comparison["official_solved"] & comparison["local_solved"]).sum()),
        int((~comparison["official_solved"] & ~comparison["local_solved"]).sum()),
    ]
})
display(solved_breakdown)

# 10. Summary Statistics

Consolidated table summarizing key evaluation metrics.

In [ ]:
summary_table = pd.DataFrame({
    "Metric": [
        "Total Instances",
        "Matches",
        "Mismatches",
        "Agreement (%)",
        "Official Solved",
        "Local Solved",
        "Mean Runtime Diff (s)",
        "Median Runtime Diff (s)"
    ],
    "Value": [
        instances,
        matches,
        mismatches,
        f"{agreement_pct:.1f}%",
        int(comparison["official_solved"].sum()),
        int(comparison["local_solved"].sum()),
        f"{comparison['runtime_difference'].mean():+.3f}s",
        f"{comparison['runtime_difference'].median():+.3f}s"
    ]
})
display(summary_table)

# 11. Visualizations

Visual summary displaying agreement percentage and cumulative solved-over-time benchmark curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Agreement Pie Chart
axes[0].pie(
    [matches, mismatches],
    labels=["Match", "Mismatch"],
    autopct="%1.1f%%",
    colors=["#4C78A8", "#F58518"],
    startangle=90
)
axes[0].set_title("Verification Agreement", fontweight="bold")

# Cumulative Solved-over-Time Curve
off_times = comparison[comparison["official_solved"]]["total_time_official"].sort_values().values
loc_times = comparison[comparison["local_solved"]]["total_time_local"].sort_values().values

axes[1].step(off_times, np.arange(1, len(off_times) + 1), where="post", label="Official", linewidth=2)
axes[1].step(loc_times, np.arange(1, len(loc_times) + 1), where="post", label="Local", linewidth=2, linestyle="--")
axes[1].set_xlabel("Runtime (s)")
axes[1].set_ylabel("Cumulative Solved Instances")
axes[1].set_title("Cumulative Solved-over-Time", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

# 12. Conclusions & Observations

## Key Observations

- **High Reproducibility:** Reached **93.2% verification agreement** across 147 CIFAR2020 instances.
- **Divergence Pattern:** All disagreements correspond to timeout/unknown cases where local execution timed out (e.g. 300s timeout vs official SAT/timeout differences). No direct SAT $\leftrightarrow$ UNSAT contradictions were observed.
- **Runtime Efficacy:** Local execution shows consistent behavior with official benchmarks, with small runtime variance on hard instances.